# Import

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import MarkdownHeaderTextSplitter

from sentence_transformers import SentenceTransformer

from pypdf  import PdfReader
from pathlib import Path
import chromadb
import unicodedata

"Create README, Create venv using .toml file, add /data to .gitignore". 

# Function

In [12]:

def clean_collection_name(name):
    # Normaliser le nom pour enlever les accents
    name = unicodedata.normalize('NFD', name).encode('ascii', 'ignore').decode('utf-8')
    # Remplacer les espaces et les tirets par des underscores
    name = name.replace(' ', '_').replace('-', '_')
    return name

In [13]:

def list_pdf_files(folder_path: str) -> list[str]:
    """Return a list of all PDF file paths in the given folder."""
    return [str(p) for p in Path(folder_path).glob("*.pdf")]

def extract_text(pdf_path: str) -> str:
    pdf_reader = PdfReader(pdf_path)
    text_content = ""
    for page in pdf_reader.pages:
        text_content += page.extract_text()
        
    print(f"Total pages: {len(pdf_reader.pages)}")
    return text_content


def chunk_text(text):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    chunks = text_splitter.split_text(text)
    
    print(f"Total text length: {len(text)} characters")
    print(f"Number of chunks: {len(chunks)}\n")

    print("First chunk preview:\n\n"+f"{chunks[0]}\n")
    print("Chunk No 10 preview:\n\n"+f"{chunks[9]}\n")
    print("Chunk No 100 preview:\n\n"+f"{chunks[99]}\n")
    return chunks

def embedd_chunks(chunks,modele):
    vecteurs = modele.encode(chunks, show_progress_bar=True)
    print(f"Nombre de chunks vectorisés : {len(vecteurs)}")
    print(f"Dimensions de chaque vecteur : {vecteurs.shape[1]}")
    return vecteurs


def create_collection(client, collection_name):
    """Create a ChromaDB collection with the specified name and cosine similarity."""
    return client.get_or_create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}
    )


def store_chunks(collection, chunks, vecteurs, collection_name):
    collection.add(
        ids=[f"chunk_{i}" for i in range(len(chunks))],
        documents=chunks,
        embeddings=vecteurs.tolist(),
        metadatas=[
            {
                "source": collection_name,
                "chunk_index": i
            }
            for i in range(len(chunks))
        ]
    )

    print(f"✅ {collection.count()} chunks stockés dans ChromaDB")

# Execute code

## Create database

In [14]:
# import 
pdf_files=list_pdf_files("P:/RAG/data/light_data")
# modele 
modele = SentenceTransformer('all-MiniLM-L6-v2')

# Connexion à ChromaDB en mode persistant (sauvegarde sur disque)
bdd_name = "light_vector_store"
client = chromadb.PersistentClient(path=f"../{bdd_name}")


# # Création de la collection (comme une table en SQL)
# collection = client.get_or_create_collection(
#     name="code_civil",
#     metadata={"hnsw:space": "cosine"}  # on utilise la similarité cosinus
# )


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
#lecture des pdf

for pdf_file in pdf_files:
    pdf_file_name=clean_collection_name(Path(pdf_file).stem)
    print(f"Processing {pdf_file}...")

    print("I Extracting text...")
    text = extract_text(pdf_file)

    print("II Chunking text...")
    chunks = chunk_text(text)

    print("III Embedding chunks...")
    vecteurs = embedd_chunks(chunks, modele)

    print("IV Creating collection in ChromaDB...")
    collection=create_collection(client, pdf_file_name)

    print("V Storing chunks in ChromaDB...")
    store_chunks(collection, chunks, vecteurs,pdf_file_name)
    

Processing P:\RAG\data\light_data\Code civil-1-373.pdf...
I Extracting text...
Total pages: 373
II Chunking text...
Total text length: 1229014 characters
Number of chunks: 1523

First chunk preview:

Code civil
Dernière modification: 2026-01-01
Edition : 2026-03-21
2896 articles avec 1118 liens
2977 références externes
Ce code ne contient que du droit positif français,
les articles et éléments abrogés ne sont pas inclus.
Il est recalculé au fur et à mesure des mises à jour.
Pensez à actualiser votre copie régulièrement à partir de codes.droit.org.
Ces codes ont pour objectif de démontrer l’utilité de l’ouverture des données publiques juridiques tant législatives que
jurisprudentielles. Il s’y ajoute une promotion du mouvement Open Science Juridique avec une incitation au dépôt du
texte intégral en accès ouvert des articles de doctrine venant du monde professionnel (Grande Bibliothèque du Droit) et
universitaire (HAL-CNRS).
Traitements effectués à partir des données issues des APIs Legi

Batches:   0%|          | 0/48 [00:00<?, ?it/s]

Nombre de chunks vectorisés : 1523
Dimensions de chaque vecteur : 384
IV Creating collection in ChromaDB...
V Storing chunks in ChromaDB...
✅ 1523 chunks stockés dans ChromaDB
Processing P:\RAG\data\light_data\Code général des impôts-1-373.pdf...
I Extracting text...
Total pages: 373
II Chunking text...
Total text length: 1803712 characters
Number of chunks: 2220

First chunk preview:

Code général des impôts
Dernière modification: 2026-03-14
Edition : 2026-03-14
2419 articles avec 5773 liens
818 références externes
Ce code ne contient que du droit positif français,
les articles et éléments abrogés ne sont pas inclus.
Il est recalculé au fur et à mesure des mises à jour.
Pensez à actualiser votre copie régulièrement à partir de codes.droit.org.
Ces codes ont pour objectif de démontrer l’utilité de l’ouverture des données publiques juridiques tant législatives que
jurisprudentielles. Il s’y ajoute une promotion du mouvement Open Science Juridique avec une incitation au dépôt du
texte in

Batches:   0%|          | 0/70 [00:00<?, ?it/s]

Nombre de chunks vectorisés : 2220
Dimensions de chaque vecteur : 384
IV Creating collection in ChromaDB...
V Storing chunks in ChromaDB...
✅ 2220 chunks stockés dans ChromaDB


## BAC A SABLE 

In [16]:

# Chunk the data using Markdown headers as delimiters 

# splitter = MarkdownHeaderTextSplitter(
#     headers_to_split_on=[
#         ("#", "Livre"),
#         ("##", "Partie"),
#         ("###", "Titre"),
#         ("####", "Chapitre")
#     ]
# )

# chunks_avec_meta = splitter.split_text(text_content)


# print(chunks_avec_meta[0].metadata)

In [17]:
# print(type(vecteurs))
# print(len(vecteurs))
# print(type(vecteurs[0]))
# print(len(vecteurs[0]))

In [18]:


# # Connexion à ChromaDB en mode persistant (sauvegarde sur disque)
# client = chromadb.PersistentClient(path="../light_vector_store")

# # Création de la collection (comme une table en SQL)
# collection = client.get_or_create_collection(
#     name="code_civil",
#     metadata={"hnsw:space": "cosine"}  # on utilise la similarité cosinus
# )

In [19]:
# resultats = collection.get(
#     ids=["chunk_0", "chunk_1"],
#     include=["documents", "metadatas"]
# )

# for doc, meta in zip(resultats["documents"], resultats["metadatas"]):
#     print(f"Source : {meta['source']}")
#     print(f"Texte  : {doc[:100]}...")
#     print("---")

In [20]:
# from retriever import rechercher, formater_contexte

# chunks = rechercher("Quels sont les droits du locataire ?")
# print(formater_contexte(chunks))

## RAG TEST

In [21]:
collections = client.list_collections()
for col in collections:
    print(col.name)

code_civil
Code_civil_1_373
Code_general_des_impots_1_373


In [22]:
# ─────────────────────────────────────────────
# Connexion à la BDD existante (lecture seule)
# ─────────────────────────────────────────────
def rechercher(question: str, k: int = 4) -> list[dict]:
    """
    Vectorise la question et retourne les k chunks les plus pertinents.

    Retourne une liste de dicts :
    [
        {"texte": "...", "source": "Code civil", "chunk_index": 42, "score": 0.87},
        ...
    ]
    """
    # 1. Vectoriser la question avec le même modèle
    vecteur_question = modele.encode(question).tolist()

    # 2. Recherche par similarité cosinus dans ChromaDB
    resultats = collection.query(
        query_embeddings=[vecteur_question],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )

    # 3. Formater les résultats
    chunks_pertinents = []
    for i in range(len(resultats["documents"][0])):
        chunks_pertinents.append({
            "texte": resultats["documents"][0][i],
            "source": resultats["metadatas"][0][i].get("source", "inconnu"),
            "chunk_index": resultats["metadatas"][0][i].get("chunk_index", i),
            "score": round(1 - resultats["distances"][0][i], 3)  # distance → similarité
        })

    return chunks_pertinents


def formater_contexte(chunks: list[dict]) -> str:
    """
    Formate les chunks en contexte lisible pour le prompt Mistral.
    """
    contexte = ""
    for i, chunk in enumerate(chunks, 1):
        contexte += f"[Extrait {i} — {chunk['source']} — pertinence : {chunk['score']}]\n"
        contexte += chunk["texte"] + "\n\n"
    return contexte.strip()


In [ ]:
test="Quand est on indigne de succéder dans le code civil?"
test2="J'ai entendu dans une conversation une histoire d'entrepreneur et de revitalisation rurale et d'amortisemtnde ses biens, aide moi a retrouver l'article qui en parle pour que je comprenne mieux "
test3="Quels sont les droits du locataire ?"

In [28]:
chunks = rechercher(test2)
chunks

[{'texte': "le fonctionnement de l'Union européenne aux aides de minimis.\nVI. – L'exonération reste applicable pour sa durée restant à courir lorsque la commune d'implantation de\nl'entreprise sort de la liste des communes classées en zone de revitalisation rurale après la date de sa création\nou de sa reprise.\n44 quindecies A.   LOI n°2026-103 du 19 février 2026 - art. 50 (V)      \n  Legif.   \n  Plan   \n  Jp.Judi.   \n  Jp.Admin.   \n  Juricaf\nI. - A. - Les contribuables qui, entre le 1er janvier 2025 et le 31 décembre 2029, créent ou reprennent des\nactivités industrielles, commerciales ou artisanales, au sens de l'article 34, ou professionnelles, au sens du 1 de\nl'article 92, dans les zones France ruralités revitalisation “ plus ” définies au III du présent article sont exonérés\nd'impôt sur le revenu ou d'impôt sur les sociétés au titre des bénéfices provenant des activités implantées dans",
  'source': 'Code_general_des_impots_1_373',
  'chunk_index': 673,
  'score': 0.625}

In [29]:
print(formater_contexte(chunks))

[Extrait 1 — Code_general_des_impots_1_373 — pertinence : 0.625]
le fonctionnement de l'Union européenne aux aides de minimis.
VI. – L'exonération reste applicable pour sa durée restant à courir lorsque la commune d'implantation de
l'entreprise sort de la liste des communes classées en zone de revitalisation rurale après la date de sa création
ou de sa reprise.
44 quindecies A.   LOI n°2026-103 du 19 février 2026 - art. 50 (V)      
  Legif.   
  Plan   
  Jp.Judi.   
  Jp.Admin.   
  Juricaf
I. - A. - Les contribuables qui, entre le 1er janvier 2025 et le 31 décembre 2029, créent ou reprennent des
activités industrielles, commerciales ou artisanales, au sens de l'article 34, ou professionnelles, au sens du 1 de
l'article 92, dans les zones France ruralités revitalisation “ plus ” définies au III du présent article sont exonérés
d'impôt sur le revenu ou d'impôt sur les sociétés au titre des bénéfices provenant des activités implantées dans

[Extrait 2 — Code_general_des_impots_1_373 — 